# TP 0 — Découvrir Spark, SQL et Parquet
**Durée indicative : 35–50 min**

Objectifs : lire un CSV, inspecter un schéma, filtrer en SQL, écrire et relire du Parquet.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


## 1. Charger les données
Le CSV de démonstration représente 4 employés fictifs. À votre avis, `Salary` est-il une chaîne ou un nombre après une lecture CSV ?

In [1]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


In [2]:
df = read_small_csv("employes_demo.csv")
df.show()
df.printSchema()

+-------+----------+---------+--------+------+
|Country|EmployeeID|FirstName|LastName|Salary|
+-------+----------+---------+--------+------+
|    USA|         1|    James|   Smith| 20000|
|     FR|         2|  Michael|    Rose| 40000|
|     FR|         3|   Robert|Williams| 30000|
|    USA|         4|    Maria|   Jones| 35000|
+-------+----------+---------+--------+------+

root
 |-- Country: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: string (nullable = true)



**✅ Réponse — `Salary` après lecture du CSV**

`Salary` est une **chaîne de caractères** (`string`), comme toutes les autres colonnes, y compris `EmployeeID` : `printSchema()` affiche `string` partout. Un CSV est un simple fichier texte qui ne contient aucune information de type : `csv.DictReader` renvoie chaque valeur sous forme de texte (`'20000'`), donc Spark en déduit `StringType`. Ce serait pareil avec `spark.read.csv(..., header=True)` sans `inferSchema` ni schéma fourni.

À noter : les colonnes s'affichent **par ordre alphabétique** (`Country`, `EmployeeID`, `FirstName`…) et non dans l'ordre du fichier, car `spark.createDataFrame` trie les clés quand on lui passe des dictionnaires.

## 2. Corriger le type et poser une question en SQL
**Question :** quels employés gagnent strictement plus de 30 000 ? Pourquoi le cast est-il utile ?

In [3]:
df = df.withColumn("Salary", F.col("Salary").cast("integer"))
df.createOrReplaceTempView("employes")
spark.sql("SELECT FirstName, Country, Salary FROM employes WHERE Salary > 30000").show()

+---------+-------+------+
|FirstName|Country|Salary|
+---------+-------+------+
|  Michael|     FR| 40000|
|    Maria|    USA| 35000|
+---------+-------+------+



In [4]:
# Complément : le schéma après le cast, puis pourquoi un texte ne se compare pas comme un nombre
df.printSchema()
spark.sql("SELECT '9000' > '30000' AS comparaison_texte, 9000 > 30000 AS comparaison_nombre").show()

root
 |-- Country: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: integer (nullable = true)

+-----------------+------------------+
|comparaison_texte|comparaison_nombre|
+-----------------+------------------+
|             true|             false|
+-----------------+------------------+



**✅ Réponse — qui gagne plus de 30 000 ? À quoi sert le cast ?**

Deux employés gagnent **strictement** plus de 30 000 : **Michael Rose (FR, 40 000)** et **Maria Jones (USA, 35 000)**. Robert Williams (30 000 pile) est exclu car `>` est strict (il faudrait `>=`), James Smith (20 000) aussi.

Le cast en `integer` est utile parce que :
- **un texte se compare caractère par caractère** : `'9000' > '30000'` vaut `true` (le `9` vient après le `3`), alors que `9000 > 30000` vaut `false`. Les tris (`ORDER BY`) et `MIN` / `MAX` sur du texte suivent ce même ordre alphabétique ;
- sans cast, Spark ferait une **conversion implicite** à chaque requête (on verrait `cast(Salary as bigint) > 30000` dans le plan) : on dépend de règles qu'on ne maîtrise pas, et en mode ANSI (actif par défaut sur Databricks et Spark 4) une valeur mal formée comme `'30 000'` ferait planter la requête ;
- les **calculs** se font sur de vrais nombres : un `SUM` sur la colonne texte renvoie un `double` (125000.0) et non un entier ;
- le **type est conservé** à l'écriture : en Parquet, `Salary` sera stocké comme entier (vérifié en partie 4).

## 3. À vous de jouer
Afficher le nombre de collaborateurs par pays avec SQL (`GROUP BY`).

In [5]:
# TODO complété : nombre de collaborateurs par pays
spark.sql("""
    SELECT Country, COUNT(*) AS nb
    FROM employes
    GROUP BY Country
    ORDER BY nb DESC, Country
""").show()

+-------+---+
|Country| nb|
+-------+---+
|     FR|  2|
|    USA|  2|
+-------+---+



**✅ Réponse**

**2 collaborateurs en France (FR)** et **2 aux États-Unis (USA)**. `GROUP BY Country` forme un groupe par pays, puis `COUNT(*)` compte les lignes de chaque groupe. J'ai ajouté un `ORDER BY` : sans lui, l'ordre des groupes n'est pas garanti (calcul distribué) et peut changer d'une exécution à l'autre.

## 4. Écrire et relire du Parquet
⚠️ Choisissez un sous-dossier personnel dans le volume. Ne jamais écraser la racine d’un volume partagé.

In [6]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
import os, re
from pathlib import Path

# Identifiant personnel déduit du compte connecté (ex. "jean.dupont@episen.fr" -> "jean_dupont") :
# chaque étudiant écrit dans son propre sous-dossier, jamais à la racine du volume partagé.
IDENTIFIANT = re.sub(r"[^a-z0-9]+", "_", spark.sql("SELECT current_user()").first()[0].split("@")[0].lower()).strip("_")

if "DATABRICKS_RUNTIME_VERSION" in os.environ or str(Path.cwd()).startswith("/Workspace"):
    VOLUME = "/Volumes/workspace/default/episen_volume"
    assert os.path.isdir(VOLUME), f"Volume introuvable : {VOLUME} (le créer dans Catalog, ou adapter le chemin)"
    OUTPUT_ROOT = f"{VOLUME}/{IDENTIFIANT}"
else:
    # Exécution hors Databricks (Spark installé en local) : dossier output/ du projet, ignoré par Git
    OUTPUT_ROOT = str(Path.cwd().parent / "output" / IDENTIFIANT)

assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."
print("Dossier de sortie :", OUTPUT_ROOT)

Dossier de sortie : /tmp/EPISEN_TP_1/output/etudiant


In [7]:
parquet_path = OUTPUT_ROOT.rstrip("/") + "/tp00_employes_parquet"
df.write.mode("overwrite").parquet(parquet_path)
df2 = spark.read.parquet(parquet_path)
df2.printSchema()
df2.show()

root
 |-- Country: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: integer (nullable = true)



+-------+----------+---------+--------+------+
|Country|EmployeeID|FirstName|LastName|Salary|
+-------+----------+---------+--------+------+
|     FR|         3|   Robert|Williams| 30000|
|    USA|         4|    Maria|   Jones| 35000|
|    USA|         1|    James|   Smith| 20000|
|     FR|         2|  Michael|    Rose| 40000|
+-------+----------+---------+--------+------+



In [8]:
# Complément : un « fichier » Parquet est en réalité un dossier
for f in sorted(Path(parquet_path).iterdir()):
    if not f.name.startswith("."):  # ignore les fichiers techniques cachés (sommes de contrôle .crc en local)
        print(f"{f.stat().st_size:>6} octets  {f.name}")
print(f"\nPour comparaison, le CSV d'origine : {(DATA_DIR / 'employes_demo.csv').stat().st_size} octets")

     0 octets  _SUCCESS
  1455 octets  part-00000-77c47267-b66a-41bb-8bae-123344bdaca4-c000.snappy.parquet
  1463 octets  part-00001-77c47267-b66a-41bb-8bae-123344bdaca4-c000.snappy.parquet

Pour comparaison, le CSV d'origine : 149 octets


**✅ Constat**

- Relu depuis le Parquet, `Salary` est toujours un **`integer`** : le schéma et les types sont enregistrés dans le fichier, alors que depuis le CSV tout revenait en `string`.
- Le résultat n'est pas un fichier mais un **dossier** : un fichier `part-….parquet` par partition du DataFrame, plus un marqueur `_SUCCESS` qui indique que l'écriture s'est bien terminée (sur Databricks, d'autres fichiers techniques comme `_committed_…` ou `_started_…` peuvent apparaître).
- Sur 4 lignes, le Parquet est **plus lourd** que le CSV : les métadonnées (schéma, statistiques) pèsent plus que les données. L'intérêt du format apparaît sur de gros volumes.

### Bilan
- Quelle différence entre **CSV** et **Parquet** ?
- Qu’est-ce qu’une **vue temporaire SQL** ?
- Dans un DataFrame, que signifie un **type** ?

**✅ Réponses du bilan**

**1. Quelle différence entre CSV et Parquet ?**

| | CSV | Parquet |
|---|---|---|
| Nature | texte lisible par un humain | binaire |
| Organisation | ligne par ligne | **colonne par colonne** (format colonnaire) |
| Schéma et types | aucun : tout est texte, il faut caster | **stockés dans le fichier** |
| Taille | pas de compression | compression et encodage par colonne |
| Lecture analytique | il faut tout lire | seulement les colonnes utiles, et des statistiques min/max permettent de sauter des blocs |
| Usage typique | échange, export, petits fichiers | stockage analytique d'un Data Lake (Delta Lake repose dessus) |

Vu dans ce TP : `Salary` revient en `string` depuis le CSV, en `integer` depuis le Parquet.

**2. Qu'est-ce qu'une vue temporaire SQL ?**

`df.createOrReplaceTempView("employes")` donne un **nom** à un DataFrame pour pouvoir l'interroger en SQL (`FROM employes`). Aucune donnée n'est copiée ni écrite : la vue ne garde que le plan de calcul, réévalué à chaque requête. Elle n'existe que dans la **session Spark en cours** et disparaît à sa fermeture, contrairement à une table enregistrée dans le catalogue (Unity Catalog). « Replace » : si le nom existe déjà, la vue est remplacée.

**3. Dans un DataFrame, que signifie un type ?**

Chaque colonne a un **type** fixé par le schéma (`string`, `integer`, `date`, `double`…), visible avec `printSchema()`. Il détermine les opérations possibles et leur sens : comparaison numérique ou alphabétique, calcul sur des dates, somme… Spark vérifie les types **avant** d'exécuter (dès l'analyse de la requête) et les conserve dans les formats typés comme Parquet. `nullable = true` signifie que la colonne peut contenir des `NULL`.